# Laser Off Code

In [ ]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

# Import

In [3]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
initialise_or_create_database_at("./2026-03-10_SNSPD4.db")
from functions import osc_set_standard, osc_check_standard, capture_trace, capture_trace_simple
from functions import snspd_dark_counts
from functions import snspd_counts_vs_wavelength
from functions import laser_set_standard, laser_get_standard
import snspd
params = snspd.snspd()

# Set up experiment
exp_name = 'SNSPD4_23_03_2026'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

Experiment loaded. Last ID no: 465


# Instruments

In [10]:
station = Station(config_file="friesland.yaml")
dmm = station.load_instrument("dmm", revive_instance=True)
yoko = station.load_instrument("yoko", revive_instance=True)
laser = station.load_instrument("laser", revive_instance=True)
MS = station.load_instrument("mso5", revive_instance=True)
pm100usb = station.load_instrument("pm100usb", revive_instance=True)
pms120 = station.load_instrument("pms120", revive_instance=True)
tc = station.load_instrument("fridge", revive_instance=True)
p_att = station.load_instrument("dmm_keithley", revive_instance=True) # excluding from snapshot because none of the parameters work anyway

# Light Counts

In [1]:
v_attenuator = 4.7 

In [11]:
osc_set_standard(MS, v_scale=params.v_scale_counts, h_time=params.h_time_counts, h_pos=params.h_pos_counts)
osc_check_standard(MS)

MANUAL
RECORDLENGTH
625000000.0
10.0
0.01
0.0
0.15
50.0
1000000000.0
0.0
0.0
0.0
1.0
0
0.0


In [12]:
print(params.v_scale_counts)
print(params.h_time_counts)
print(params.h_pos_counts)

0.15
0.1
0


In [17]:
wavelength_range = np.arange(1528e-9, 1565e-9, 1e-9) # wavelength range of PPCL550
v_attenuator = 4.7 # from SNSPD4-2_Calibration.ipynb ID 457

# Set oscilloscope vertical and horizontal parameters 
osc_set_standard(MS, v_scale=params.v_scale_counts, h_time=params.h_time_counts, h_pos=params.h_pos_counts)
# osc_check_standard(MS)

if MS.channels[0].clipping(): 
    print('Error: Clipping')

# Set attenuator voltage
p_att.write(f'VOLT {v_attenuator}')
time.sleep(10)

# Set standard laser parameters 
laser_set_standard(laser, wavelength_range[0], 7)
laser_get_standard(laser)

# Set current 
yoko.current(12e-6)

# ############################ TURN LASER ON ############################ 
# laser.enable(True)
# print(f'Laser enable status: {laser.enable()}')

# time.sleep(10)

# if MS.channels[0].clipping(): 
#     print('Error: Clipping')

# snspd_counts_vs_wavelength(MS, dmm, yoko, p_att, laser, device_name, n_captures, interval, wavelength_range, station=station)

# ############################ TURN LASER OFF ############################ 
# laser.enable(False)
# print(f'Laser enable status: {laser.enable()}')

Power: 7.0
Frequency coarse: 196.1992THz
Wavelength (calculated) is 1528.0004097876035nm


In [18]:
laser.enable()

False